# Notebook 01 — Extracción y Unión de Datos
**NHANES 2013-2014 · Ciclo H · Autor: Nicholas**

**Objetivo:** Descargar las tablas clave del CDC, unirlas por `SEQN` y guardar
la tabla maestra en `data/01_raw/nhanes_2013_maestra.csv`.

**Data Augmentation histórico:** Para balancear la clase minoritaria (longevos ≥70 años),
se incorporan pacientes ≥70 del ciclo 2015-2016 y de los ciclos anteriores 2009-2010,
2007-2008 y 2005-2006.

**Tablas descargadas por ciclo:**
`DEMO`, `BMX`, `BPX`, `TCHOL`, `GLU`, `MCQ`, `SMQ`

---

## 1. Importación de librerías

In [1]:
%pip install pandas pyreadstat requests pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import io, gzip, warnings, os, json
from pathlib import Path

import pandas as pd
import numpy as np
import requests

warnings.filterwarnings('ignore')

# ─── Header que simula un navegador real ─────────────────────────────────────
# El CDN del CDC bloquea requests sin User-Agent y devuelve HTML en lugar del .xpt
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    )
}

BASE_DIR = Path('../data/01_raw')
BASE_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas correctamente.")
print(f"Directorio de salida: {BASE_DIR.resolve()}")

Librerías cargadas correctamente.
Directorio de salida: C:\Users\nholc\Desktop\ev3_nanhes\data\01_raw


## 2. Configuración de ciclos

**Ciclo base:** 2013-2014 (letra H) — todos los pacientes.

**Ciclos históricos (solo longevos ≥70):**
| Ciclo | Letra |
|---|---|
| 2015-2016 | I |
| 2009-2010 | F |
| 2007-2008 | E |
| 2005-2006 | D |

> El truco pro: en los ciclos históricos se aplica un filtro de edad **antes** de unir
> las tablas de laboratorio, ahorrando RAM y evitando introducir pacientes jóvenes.

In [2]:
# Ciclo que este grupo analiza
_CICLO_BASE = {"año": "2013", "letra": "H", "nombre": "2013-2014"}

# Ciclos adicionales para rescatar longevos (clase minoritaria)
_CICLOS_HISTORICOS = [
    {"año": "2015", "letra": "I",  "nombre": "2015-2016"},
    {"año": "2009", "letra": "F",  "nombre": "2009-2010"},
    {"año": "2007", "letra": "E",  "nombre": "2007-2008"},
    {"año": "2005", "letra": "D",  "nombre": "2005-2006"},
]

_TABLAS_CLAVE = ["DEMO", "BMX", "BPX", "TCHOL", "GLU", "MCQ", "SMQ"]
_EDAD_LONGEVO = 70

def _generar_url(tabla: str, año: str, letra: str) -> str:
    """URL pública del CDC para archivos .xpt de NHANES."""
    return (
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{año}/"
        f"DataFiles/{tabla}_{letra}.xpt"
    )

print("Configuración lista.")
print(f"Ciclo base : {_CICLO_BASE['nombre']}")
print(f"Ciclos hist: {[c['nombre'] for c in _CICLOS_HISTORICOS]}")

Configuración lista.
Ciclo base : 2013-2014
Ciclos hist: ['2015-2016', '2009-2010', '2007-2008', '2005-2006']


## 3. Funciones de descarga

In [4]:
SAS_SENTINEL = 5.397605e-79

def _limpiar_missing_sas(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte el valor centinela SAS (5.4e-79) a NaN verdadero."""
    cols_num = df.select_dtypes(include=[np.number]).columns
    for col in cols_num:
        mask = np.isclose(df[col], SAS_SENTINEL, atol=1e-85) & (df[col] > 0)
        df.loc[mask, col] = np.nan
    return df


def _descargar_ciclo(config_ciclo: dict, solo_longevos: bool = False) -> pd.DataFrame:
    """
    Descarga las tablas de un ciclo NHANES desde el CDC y las une por SEQN.
    Si solo_longevos=True, filtra a ≥70 años justo después de bajar DEMO
    para ahorrar RAM.
    """
    año   = config_ciclo["año"]
    letra = config_ciclo["letra"]
    nombre = config_ciclo["nombre"]

    print(f"\n--- Descargando Ciclo {nombre} ---")

    # 1. Tabla maestra DEMO
    url_demo = _generar_url("DEMO", año, letra)
    try:
        resp = requests.get(url_demo, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        if 'html' in resp.headers.get('Content-Type', '').lower():
            raise ValueError("CDC devolvió HTML en lugar del XPT")
        import pyreadstat
        df_maestra, _ = pyreadstat.read_xport(io.BytesIO(resp.content))
    except Exception as e:
        print(f"  ✗ Error descargando DEMO para {nombre}: {e}")
        return pd.DataFrame()

    df_maestra = _limpiar_missing_sas(df_maestra)

    # 2. Filtro temprano para ciclos históricos
    if solo_longevos:
        if 'RIDAGEYR' in df_maestra.columns:
            df_maestra = df_maestra[df_maestra['RIDAGEYR'] >= _EDAD_LONGEVO].copy()
            print(f"  Rescatados {len(df_maestra)} pacientes longevos de {nombre}.")
        else:
            print(f"  ✗ RIDAGEYR no encontrada en {nombre}. Skipping.")
            return pd.DataFrame()

    # 3. Resto de tablas
    for tabla in _TABLAS_CLAVE:
        if tabla == "DEMO":
            continue
        url = _generar_url(tabla, año, letra)
        try:
            resp = requests.get(url, headers=HEADERS, timeout=60)
            resp.raise_for_status()
            if 'html' in resp.headers.get('Content-Type', '').lower():
                raise ValueError("CDC devolvió HTML")
            import pyreadstat
            df_temp, _ = pyreadstat.read_xport(io.BytesIO(resp.content))
            df_temp = _limpiar_missing_sas(df_temp)

            # LEFT join para ciclos históricos (evita traer jóvenes de laboratorio)
            tipo_join = 'left' if solo_longevos else 'outer'
            df_maestra = pd.merge(df_maestra, df_temp, on='SEQN', how=tipo_join)
            print(f"  ✓ {tabla}_{letra} integrada.")
        except Exception as e:
            print(f"  ✗ No se encontró {tabla}_{letra}: {e}")

    df_maestra['CICLO_ORIGEN'] = nombre
    return df_maestra

## 4. Descarga del ciclo base 2013-2014 + Data Augmentation histórico

In [5]:
def descargar_y_unir_2013() -> pd.DataFrame:
    """
    Nodo de Kedro: descarga el ciclo 2013-2014 completo y añade
    pacientes ≥70 de los ciclos históricos para balancear la clase minoritaria.
    """
    print("Iniciando extracción con Data Augmentation histórico...")

    # 1. Ciclo base — todos los pacientes
    df_final = _descargar_ciclo(_CICLO_BASE, solo_longevos=False)

    # 2. Ciclos históricos — solo longevos
    for ciclo in _CICLOS_HISTORICOS:
        df_hist = _descargar_ciclo(ciclo, solo_longevos=True)
        if not df_hist.empty:
            df_final = pd.concat([df_final, df_hist], ignore_index=True)

    print(f"\n✓ Extracción completada. Dataset robustecido: {len(df_final):,} pacientes")
    return df_final


# Ejecutar
df_raw = descargar_y_unir_2013()

Iniciando extracción con Data Augmentation histórico...

--- Descargando Ciclo 2013-2014 ---
  ✓ BMX_H integrada.
  ✓ BPX_H integrada.
  ✓ TCHOL_H integrada.
  ✓ GLU_H integrada.
  ✓ MCQ_H integrada.
  ✓ SMQ_H integrada.

--- Descargando Ciclo 2015-2016 ---
  Rescatados 952 pacientes longevos de 2015-2016.
  ✓ BMX_I integrada.
  ✓ BPX_I integrada.
  ✓ TCHOL_I integrada.
  ✓ GLU_I integrada.
  ✓ MCQ_I integrada.
  ✓ SMQ_I integrada.

--- Descargando Ciclo 2009-2010 ---
  Rescatados 1112 pacientes longevos de 2009-2010.
  ✓ BMX_F integrada.
  ✓ BPX_F integrada.
  ✓ TCHOL_F integrada.
  ✓ GLU_F integrada.
  ✓ MCQ_F integrada.
  ✓ SMQ_F integrada.

--- Descargando Ciclo 2007-2008 ---
  Rescatados 1155 pacientes longevos de 2007-2008.
  ✓ BMX_E integrada.
  ✓ BPX_E integrada.
  ✓ TCHOL_E integrada.
  ✓ GLU_E integrada.
  ✓ MCQ_E integrada.
  ✓ SMQ_E integrada.

--- Descargando Ciclo 2005-2006 ---
  Rescatados 882 pacientes longevos de 2005-2006.
  ✓ BMX_D integrada.
  ✓ BPX_D integrada.
  ✓

## 5. Inspección rápida

In [6]:
print(f"Shape: {df_raw.shape}")
print(f"Nulos totales: {df_raw.isnull().sum().sum():,}")
print(f"Ciclos en CICLO_ORIGEN: {df_raw['CICLO_ORIGEN'].value_counts().to_dict()}")
df_raw.head(3)

Shape: (14276, 298)
Nulos totales: 2,852,216
Ciclos en CICLO_ORIGEN: {'2013-2014': 10175, '2007-2008': 1155, '2009-2010': 1112, '2015-2016': 952, '2005-2006': 882}


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,INDHHINC,INDFMINC,DMDHRBRN,BMXCALF,BMICALF,BMXTHICR,BMITHICR,BPXDB,SMD070,SMD075
0,73557.0,8.0,2.0,1.0,69.0,NaN,4.0,4.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,73558.0,8.0,2.0,1.0,54.0,NaN,3.0,3.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,73559.0,8.0,2.0,1.0,72.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Guardado en disco

In [7]:
OUTPUT_PATH = BASE_DIR / 'nhanes_2013_maestra.csv'
df_raw.to_csv(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1024**2
print(f"✓ Guardado en: {OUTPUT_PATH}")
print(f"  Tamaño: {size_mb:.2f} MB | {len(df_raw):,} filas × {df_raw.shape[1]} cols")

✓ Guardado en: ..\data\01_raw\nhanes_2013_maestra.csv
  Tamaño: 8.91 MB | 14,276 filas × 298 cols


## 7. Conclusiones

- El ciclo **2013-2014** aporta la totalidad de los pacientes base.
- Los ciclos históricos inyectan pacientes longevos ≥70 años para equilibrar la clase
  minoritaria antes del modelado.
- El archivo `nhanes_2013_maestra.csv` es el input del Notebook 02.

---
*Siguiente paso → Notebook 02: Preprocesamiento*